# ¿Aporta algo Wan 2.2?

El vídeo generativo está **en evaluación** (§20 del plan). El criterio de adopción no es "qué bonito
queda", es concreto: *que un clip en el gancho mejore la retención frente a la versión solo con Manim*.
Y hay dos reglas: **nada de texto generado y nada de personas realistas**.

Así que los seis planos de aquí abajo no son demos abstractas: son los planos que el episodio 00
podría usar de verdad. El gancho habla de un invernadero, dos sensores y un hongo —cosas del mundo
físico que Manim no sabe dibujar— mientras la pantalla enseña ejes y puntos. Ahí es donde Wan tendría
algo que aportar, si es que lo aporta.

Esto va a calidad de publicación: 1280x704, 50 pasos, unos **5 minutos de GPU por clip** más la carga
inicial del modelo. Media hora larga en total, y se puede interrumpir: los clips ya hechos se saltan.

**Antes de empezar**, el entorno de los modelos generativos (una vez):

```bash
./env/setup_qwen.sh
```

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

RAIZ = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "pipeline" / "produce.py").exists()), None)
if RAIZ is None:
    raise RuntimeError("Abre el notebook desde dentro de series-redes-neuronales/")
if Path(sys.prefix).resolve() != (RAIZ / ".venv").resolve():
    raise RuntimeError(f"Kernel equivocado: {sys.executable}. "
                       f"Selecciona 'series-redes-neuronales (.venv 3.13)' o {RAIZ / '.venv/bin/python'}")

import torch
from IPython.display import Video, display

PYTHON_QWEN = RAIZ / ".venv-qwen" / "bin" / "python"
if not PYTHON_QWEN.exists():
    raise RuntimeError("Falta .venv-qwen: ejecuta ./env/setup_qwen.sh")

# Los pesos vienen en float32 (32 GB) y cargarlos pide unos 22 GB de RAM. Con el límite de 24 GB que
# traía .wslconfig, el kernel mataba el proceso a mitad de carga.
ram = int(next(l.split()[1] for l in open("/proc/meminfo") if l.startswith("MemTotal"))) / 2**20
if ram < 40:
    raise RuntimeError(
        f"WSL solo ve {ram:.0f} GB de RAM y la carga necesita ~22 GB con holgura.\n"
        "En C:\\Users\\amio0\\.wslconfig debe poner memory=48GB; después, en PowerShell:\n"
        "    wsl --shutdown\n"
        "y vuelve a abrir WSL (se cierra todo lo que tengas abierto dentro).")

SALIDA = RAIZ / "build" / "broll"
SALIDA.mkdir(parents=True, exist_ok=True)
print(f"{RAIZ} · {torch.cuda.get_device_name(0)} · {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB VRAM"
      f" · {ram:.0f} GB RAM")

## 1. Los planos candidatos

Cada uno tiene un sitio en el episodio; si un clip no tiene sitio, no se juzga, se tira.

| id | Dónde iría | Qué tendría que conseguir |
|---|---|---|
| `invernadero_noche` | s01, bajo las primeras frases del gancho | Que el invernadero exista antes de que aparezca el gráfico |
| `sensores` | s01, en «temperatura, humedad» | Enseñar de dónde salen los dos ejes |
| `hongo_macro` | s01, en «avisar de un hongo» | Que la consecuencia de fallar se vea, no se explique |
| `dos_lunas` | s01→s02, la transición al gráfico | Un puente del mundo real a los datos que en Manim sería un corte seco |
| `invernaderos_vecinos` | s02, «en el de al lado, con otro cultivo» | Que se entienda que el problema se repite y cambia |
| `senales_red` | cierre, bajo la última frase | Un plano de respiración, sin información nueva |

Los prompts van en inglés a propósito: el codificador de texto de Wan es multilingüe, pero la
inmensa mayoría de sus descripciones de entrenamiento están en inglés y la obediencia al prompt es
notablemente mejor. La estructura que funciona es siempre la misma: **sujeto → acción → cámara →
luz → estilo**, con el movimiento dicho de forma explícita y lenta.

Los colores no son decorativos: azul para los datos (`#4C9BE8`), naranja para la otra clase
(`#F2A541`), violeta para los pesos y verde azulado para las activaciones, como en `config/estilo.yaml`.
El negativo común (texto, personas, deformaciones) ya vive en `pipeline/wan_clip.py`.

In [ ]:
CLIPS = [
 {"id": "invernadero_noche", "segundos": 3, "prompt":
  "Interior of a humid greenhouse at night, long rows of dark leafy plants receding into soft fog, "
  "cold blue overhead grow lights, condensation running down the glass panels, slow steady dolly "
  "forward between the rows, shallow depth of field, cinematic, dark teal and deep blue color "
  "grading, calm and slightly ominous"},

 {"id": "sensores", "segundos": 3, "prompt":
  "Close up of two small sensor probes on a thin metal stake inside a dark greenhouse, one tiny "
  "amber indicator light and one tiny blue indicator light blinking slowly, water vapor drifting "
  "past, blurred rows of plants in the background, very slow orbit around the stake, shallow depth "
  "of field, cinematic dark background"},

 {"id": "hongo_macro", "segundos": 3, "prompt":
  "Extreme macro of pale grey mould spreading across a wet green leaf, delicate fungal filaments "
  "creeping outward, tiny water droplets catching cold blue light, black background, slow push in, "
  "time lapse growth, scientific documentary look, shallow depth of field"},

 {"id": "dos_lunas", "segundos": 3, "prompt":
  "Hundreds of small glowing dots in deep dark navy space, half of them soft blue and half warm "
  "orange, drifting slowly and settling into two interleaved crescent shaped clusters, soft bokeh, "
  "subtle depth, slow camera drift, elegant minimal abstract data visualisation, dark background"},

 {"id": "invernaderos_vecinos", "segundos": 3, "prompt":
  "Slow aerial drift over three long greenhouse roofs at dusk, glass panels glowing faintly from "
  "within in different hues, thin fog lying over the fields, deep blue evening sky, cinematic, "
  "minimal, calm"},

 {"id": "senales_red", "segundos": 3, "prompt":
  "Abstract dark translucent layers slowly undulating, thin violet and teal light trails travelling "
  "across and between them like signals through a network, deep black background, very slow motion, "
  "elegant, minimal, volumetric glow"},
]

# Recorta la lista si quieres empezar por uno o dos (cada clip son ~5 min de GPU).
SELECCION = [c["id"] for c in CLIPS]
print(f"{len(SELECCION)} clips · ~{len(SELECCION) * 5 + 4} min en total, carga del modelo incluida")

## 2. Generarlos

El modelo se carga **una sola vez** para todo el lote (son 32 GB en float32; la carga sola son unos
minutos). Si interrumpes la celda, los clips terminados se conservan y la siguiente ejecución sigue
por donde iba.

In [ ]:
trabajo = SALIDA / "trabajo_wan.json"
trabajo.write_text(json.dumps({
    "salida": str(SALIDA),
    "clips": [c for c in CLIPS if c["id"] in SELECCION],
}, ensure_ascii=False, indent=1), encoding="utf-8")

# Sin HF_HUB_OFFLINE: los pesos ya están, pero diffusers comprueba la revisión al abrirlos.
entorno = {k: v for k, v in os.environ.items() if k != "HF_HUB_OFFLINE"}
inicio = time.time()
proceso = subprocess.run([str(PYTHON_QWEN), str(RAIZ / "pipeline/wan_clip.py"), "--lote", str(trabajo)],
                         cwd=RAIZ, env=entorno, capture_output=True, text=True)
print(proceso.stdout[-3000:])
if proceso.returncode != 0:
    print(proceso.stderr[-3000:])
    raise RuntimeError("Wan falló; si no hay traza de Python, fue el OOM killer (mira `dmesg | tail`)")
print(f"total: {(time.time() - inicio) / 60:.1f} min")

## 3. Verlos

Tres preguntas por clip, en este orden:

1. **¿El movimiento es creíble?** Wan falla en lo que se mueve raro: plantas que respiran, gotas que
   suben, cámaras que se aceleran solas.
2. **¿Se sostiene 3 segundos?** La coherencia se rompe casi siempre al final del clip.
3. **¿Convive con Manim?** Si el plano pide atención, compite con la explicación en vez de apoyarla.

In [ ]:
papeles = {c["id"]: c["prompt"][:70] + "…" for c in CLIPS}
for c in CLIPS:
    ruta = SALIDA / f"{c['id']}.mp4"
    if not ruta.exists():
        continue
    display(f"── {c['id']} · {papeles[c['id']]}")
    display(Video(str(ruta), embed=True, width=800))

## 4. Decidir, con el coste delante

Lo que el plan pide contestar: si el gancho con clip retiene mejor que el gancho solo con Manim.
Eso no lo dice este notebook, lo dicen las métricas del piloto; pero sí puedes decidir aquí si
alguno de estos planos es lo bastante bueno como para llevarlo al montaje y medirlo.

In [ ]:
resultado = SALIDA / "resultado.json"
filas = json.loads(resultado.read_text(encoding="utf-8")) if resultado.exists() else []
for f in filas:
    print(f"{f['id']:22} {f['segundos']:.1f}s · {f['resolucion']} · {f['minutos_generacion']:.1f} min")
if filas:
    total = sum(f["minutos_generacion"] for f in filas)
    seg = sum(f["segundos"] for f in filas)
    print(f"\n{total:.0f} min de GPU para {seg:.0f} s de vídeo → {total / seg:.1f} min por segundo")
    print(f"un episodio de 221 s con un 10% de b-roll: {total / seg * 22 / 60:.1f} h de GPU")

In [ ]:
# Notas de la visión (rellénalas tú):
NOTAS = """
Se salvan:      ...
Se caen:        ...
Por qué:        ...

¿Al montaje del gancho?  [ ] sí, y medimos retención  [ ] no
Decisión (§20 del plan): [ ] entra en la serie  [ ] se aplaza  [ ] se descarta
"""
(SALIDA / "notas.md").write_text(NOTAS, encoding="utf-8")
print(NOTAS)